# Qwen3.5-9B 高等数学解题助手 —— QLoRA 微调实验

## 一、实验目的

**主目标**：用自建的中文高等数学 SFT 数据集，通过 QLoRA 对 `Qwen/Qwen3.5-9B` 进行参数高效微调，得到一个**擅长解答中文高等数学题**的专用智能助手——在保留基座通用能力的前提下，学会规范的解题推理格式。

### 1.1 能力目标

- 正确求解 29 类高等数学题型：极限、一元微分/积分、级数、多元微积分、微分方程、几何应用等；
- 输出规范的解题轨迹：先用 `<think>...</think>` 展示推导，再给出结论，最终答案统一放入 `\boxed{}`；
- 提升中文数学表达与 LaTeX 公式书写的规范性与稳定性。

### 1.2 工程与研究目标

- 验证「单卡 RTX 4090D (24GB) + QLoRA」训练 9.65B 模型的完整链路（环境已含 flash-attn 加速）；
- 建立**可复现的对照实验**：基座模型 vs 5k-LoRA vs 20k-LoRA，在同一隔离评测集上对比；
- 沉淀训练配置与评测流程，为后续 GRPO（利用 `canonical_answer` 可自动验算的题目）打基础。

### 1.3 成功标准（初版）

| 指标 | 目标 |
|------|------|
| 评测集答案正确率（`\boxed{}` 提取 + LaTeX 等价判定） | 相对基座显著提升（≥ +15pp 为佳） |
| 格式合规率（含 think 推理块且含 `\boxed{}`） | ≥ 95% |
| 泛化检查（未见模板/变体题面） | 无明显退化，不出现"背模板"现象 |
| 训练稳定性 | 无 NaN / 损失尖峰，显存峰值 < 24 GB |

## 二、实验计划（分阶段推进）

> **数据集要点**（详见 `data/README.md`）：5k 用于试训与对照、20k 用于正式训练，两者按题面 SHA-256 互不重复；样本已含 `<think>` 推理轨迹与 `\boxed{}` 答案；单条长度 136–223 token（100% 覆盖于 1024 以内）。

### 阶段 0：准备【已完成 ✅】

- 环境：transformers 5.20 / peft 0.21 / trl 1.15 / bitsandbytes 0.50 / flash-attn 2.8.3（GPU 冒烟测试通过）；
- 模型：`/root/autodl-tmp/models/Qwen3.5-9B`（9.65B 参数，文件完整性校验通过）；
- 数据：`/root/autodl-tmp/data/`（SHA-256 与 `validation_report.json` 比对一致）。

### 阶段 1：冒烟测试【已完成 ✅ 2026-10-16】（300 条 × 60 step）

结果：loss 1.32 → 0.008（3.8 分钟）｜峰值显存 8.49 GiB｜可训练参数 43.3M (0.78%)｜3/3 未训样本格式合规且答案一致。详见「阶段 1 结果」。

- [x] assistant-only loss：已确认——63.1% token 参与训练，system/user 部分被正确屏蔽；
- [x] 序列构造正确：EOS（`<|im_end|>`）位于损失掩码内，生成正常收尾；
- [x] 显存峰值实测：8.49 GiB（目标 < 22 GB，余量充足）；
- [x] loss 平滑下降、无 NaN；抽样生成格式与答案全部通过。

### 阶段 2：5k 完整训练 + 对照【已完成 ✅ 2026-10-16】

- [x] 5k 全量训练 2 epochs（626 步 / 20.5 分钟 / 峰值显存 8.50 GiB / loss 1.246 → 0.0057），适配器：`qlora_math/outputs/5k/adapter`；
- [x] 与基座对照（24 条未训练样本）：格式合规 0% → **100%**，答案一致 0% → **75%**，详见「阶段 2 结果」；
- [ ] 人工抽查 50–100 条生成结果（对错、步骤质量、格式）——待补。

### 阶段 3：20k 扩展训练【已完成 ✅ 2026-10-16】

- [x] 20k 全量训练 2 epochs（2500 步 / 81.8 分钟 / loss 1.391 → 0.0051），适配器：`qlora_math/outputs/20k/adapter`；
- [x] 与 5k-LoRA 同集对比：隔离集 300 题**完全持平（37.7% vs 37.7%，逐题翻转 18↔18 净零）**——同分布数据量 4 倍未带来 OOD 泛化增益，详见「阶段 3 结果」；
- [x] 模板过拟合检查：确认 9 类 0% 题型需「模板族扩充 + 难度 3 数据 + 验算步骤样本」，已写入阶段 4 方向。

### 阶段 4：评测与迭代

- [x] 隔离评测集已就位：`data/high_math_eval_300/`（300 题，isolated_v2）与 `data/sft_v3/high_math_eval_v3/`（300 题，eval_v3）；
- [x] 评测工具链就绪：`\boxed{}` 提取 + math_verify LaTeX 等价判定 + 分题型/难度统计（`validate.py` / `rescore.py`）；
- [x] ① 数据扩充首轮（v3）完成：sft_v3 训练 + 双基准评测——**eval_v3 100%**｜iso300 34.3%（旧强项有回退）→ 详见「v3 实验结果」；
- [ ] ② **v3.1**：v3 新结构 + 全量旧 20k 均衡回放（40k），修复遗忘；验收：eval_v3 ≥ 95% 且 iso300 ≥ 40%；
- [ ] ③ GRPO 可验证奖励；④ 人工题与错题回流。

### 训练配置（阶段 1 实测版）

| 项目 | 取值 | 说明 |
|------|------|------|
| 量化 | 4-bit NF4 + double quant，bf16 计算 | QLoRA 标准配置 |
| LoRA | r=16，α=32，dropout=0.05 | target 覆盖 12 类线性层：全注意力 q/k/v/o + 线性注意力 in_proj_qkv/z/b/a、out_proj + MLP gate/up/down（实测模型为 8 层全注意力 + 24 层线性注意力混合架构） |
| max_length | 1024（实测最长 223/275，100% 覆盖） | 追求吞吐可降至 512 |
| 批大小 | per-device 8 × 梯度累积 2（等效 16） | 实测显存余量大，可加大 |
| 优化器 | paged_adamw_8bit，lr=2e-4，cosine + 3% warmup | QLoRA 常用配置 |
| 注意力实现 | flash_attention_2 + flash-linear-attention | 有效加速混合注意力层 |
| 梯度检查点 | 开启 | 以少量速度换显存 |
| 精度 | bf16 | 4090D 原生支持 |
| Loss | assistant_only_loss=True | 只对 assistant 回复（含 think 块）计损失；`<|im_end|>` 在掩码内 |

### 风险与对策

1. **数据为程序化合成**（规则校验，非逐条人工审校）→ 正式长跑前随机抽查 ≥100 条；
2. **模板同构导致"背模板"** → 评测必须模板隔离，必须加入变体题面（隔离集已暴露该风险，v3 已部分解决，仍需跨族评测）；
3. **续训遗忘** → v3 已验证旧强项回退风险，v3.1 用均衡回放修复；
4. **显存管理** → 峰值 8.5–8.7 GiB，余量充足。

### Notebook 推进路线

1. 实验目的与计划（本部分）
2. 环境与路径预检
3. 数据加载与 chat template 处理（assistant-only loss mask 验证）
4. QLoRA 配置与模型加载（4-bit + LoRA 注入）
5. 冒烟测试训练（300 条）✅
6. 5k / 20k 完整训练（5k ✅ 20k ✅）
7. 推理与隔离评测（iso300 三模型对比 ✅ / eval_v3 ✅）

## 📊 阶段 1：冒烟测试结果（2026-10-16 通过 ✅）

**配置**：5k 文件前 300 条 × 60 步 ｜ QLoRA 4bit NF4 + bf16 ｜ batch 8 × 梯度累积 2 ｜ lr 2e-4（cosine，warmup 10 步）｜ assistant-only loss ｜ flash-attn + flash-linear-attention

### 训练指标

| 指标 | 实测 | 验收目标 | 状态 |
|------|------|---------|------|
| Loss | 1.32 → 0.008（平滑，无 NaN） | 平滑下降 | ✅ |
| 峰值显存 | 8.49 GiB | < 22 GiB | ✅（余量充足） |
| 训练耗时 | 3.8 分钟（3.84 s/step） | — | ⚡ 5k ≈ 20 分钟/epoch |
| 可训练参数 | 43.3 M / 5.54 B = 0.78% | — | ✅ |
| 有效训练 token | ≈ 16.4 万（仅 assistant 段计入） | — | ✅ |

### 生成抽查（3 条未参与训练的样本，贪心解码）

| # | 题型 | 推理格式 | 答案校验 |
|---|------|---------|---------|
| 1 | 数列有理式极限 | think ✅ / boxed ✅ / EOS ✅ | ✅ 与参考答案一致（`5`） |
| 2 | 等比级数求和 | think ✅ / boxed ✅ / EOS ✅ | ✅ 一致（`-\frac{9}{2}`） |
| 3 | 一阶线性微分方程 | think ✅ / boxed ✅ / EOS ✅ | ✅ 一致（`y=\frac{37}{3}e^{-6x}-\frac{4}{3}`） |

### 关键过程发现

- 模型为**混合注意力架构**：8 层全注意力（q/k/v/o_proj）+ 24 层线性注意力（in_proj_qkv/z/b/a、out_proj），LoRA 目标层已相应覆盖 12 类；
- TRL 自带与模型原生模板逐行对应的训练模板（`qwen3_5_think_training.jinja`，含 `{% generation %}` 标记），`assistant_only_loss=True` 开箱即用，且 `<|im_end|>` 在损失掩码内；
- 已安装 `flash-linear-attention` 加速线性注意力层；`causal_conv1d` 缺失（回退慢速版，当前速度可接受）。

### 产物

- 冒烟适配器：`qlora_math/outputs/smoke/adapter`（含 `train_summary.json` 与 TensorBoard 日志）
- 训练 / 推理代码：`qlora_math/train.py`、`qlora_math/infer.py`（运行：`python train.py --mode smoke|5k|20k`）
- 适配器「保存 → 重新加载 → 生成」闭环验证通过 ✅

**结论：训练链路全部打通，可进入阶段 2（5k 全量训练 + 基座对照）。**

## 📊 阶段 2：5k 训练与对照结果（2026-10-16）

### 训练指标（5k 全量 × 2 epochs）

| 指标 | 实测 |
|------|------|
| 步数 / 耗时 | 626 步 / 20.5 分钟（1.96 s/step） |
| Loss | 1.246 → 0.0057（全期均值 0.040） |
| 峰值显存 | 8.50 GiB |
| 可训练参数 | 43.3 M / 5.54 B（0.78%） |
| 产物 | `qlora_math/outputs/5k/adapter`（含 checkpoint-600/626 与 TensorBoard 日志） |

### 基座 vs 5k-LoRA 对照（24 条未训练样本①，贪心解码，严格协议）

| 指标 | 基座 | 5k-LoRA |
|------|------|---------|
| 格式合规率（think+boxed+EOS） | 0.0% | **100.0%** |
| 答案一致率（`\boxed{}` 提取 + 严格文本匹配） | 0.0% | **75.0%（18/24）** |
| 平均输出长度 / 单条耗时 | 1518 字符 / 26 s | 175 字符 / 9.3 s |

① 取样自 20k 文件（与 5k 训练集按题面 SHA-256 互斥）；正式隔离评测集留待阶段 4。

**基座说明**：基座输出为英文长篇「thinking process」（平均 1518 字符），24 条中 **0 条**能在 512 token 内完成 think 区块并给出最终答案 → 0% 反映的是**输出协议不合规**，而非解题能力为零。这正体现本阶段目标：把能力对齐到规范的中文解题协议。

**LoRA 未命中 6 条均为真实计算错误**（已逐条核对，非格式差异）：

| # | 题型 | 生成答案 | 参考答案 |
|---|------|---------|---------|
| 7 | derivative_polynomial_at_point | -82 | -86 |
| 9 | integral_symmetry | -14928/5 | -15072 |
| 17 | tangent_line_polynomial | y-50=21(x-4) | y-58=21(x-4) |
| 19 | integral_symmetry | 15504/5 | 16548 |
| 20 | total_differential | -10dx+54dy | -14dx+54dy |
| 23 | integral_symmetry | 404/15 | 476/15 |

错误集中在 `integral_symmetry`（对称性定积分 ×3）与个别系数/算术滑移 → 预期 20k 训练与阶段 4（LaTeX 等价判定 + 错题回流 + GRPO 可验算奖励）可进一步压低。

### 结论与下一步

- ✅ 阶段 2 目标达成：未见题上 100% 格式合规 + 75% 严格答案一致（基座同协议 0%）；
- ✅ 泛化初步验证：24 条覆盖 15 类题型，格式全部保持，无「背模板」崩溃迹象；
- 下一阶段：按计划进入 **阶段 3（20k 全量训练）**；或先补阶段 4 的隔离评测集与 LaTeX 等价判定后再决策。

完整生成文本：`qlora_math/outputs/compare/base_vs_5k_n24.json`

## 📊 隔离验证集首评（300 题，2026-10-16）

> 评测集：`data/high_math_eval_300/`（用户提供）——29 类题型各 10–11 题，题面与 5k/20k 训练集 SHA-256 零重合，独立模板族（isolated_v2），难度 2/3（训练数据仅含难度 1–2）。

### 总体指标（5k-LoRA，贪心解码，max_new_tokens=1024）

| 指标 | 结果 |
|------|------|
| 格式合规率 | **100%**（300/300，0 截断） |
| 等价正确率（math_verify LaTeX 等价判定） | **37.7%（113/300）** |
| 严格文本匹配 | 31.0%（等价判定救回 20 条等价写法，如 `x^5` vs `x^{5}`、`9e^{8x}(…)` vs `e^{8x}(9…)`） |
| 难度 2 正确率 | 48.9%（92/188） |
| 难度 3 正确率 | **18.8%（21/112）** |

### 分题型：明显的两极分化

- ✅ **满分/近满分**：数列有理极限 11/11、单调区间 11/11、曲线弧长 10/10、收敛半径 10/10、代换积分 9/10、二阶常系数 ODE 8/10、不定积分 8/10；
- ❌ **零分**（8 类)：隐函数求导、指数三角求导、方向导数、对称性积分、二重积分、偏导数、一阶线性 ODE、切平面。

### 失败性质（逐条抽查完整生成）

- 难度 2 失败：仍以**计算滑移**为主（偏导代点 `-5` 算成 `-1`）；
- 难度 3 失败：出现**方法性错误**（如隐函数求导把 `6xy` 的导数写成 `6y(y'+1)`）→ 对新结构/更高难度的深层泛化不足；
- 所有失败均与格式无关——**协议能力在 OOD 数据上零退化**（300/300 合规）。

### 结论与下一步

1. 5k 模型在**同族题面**（难度 1–2）≈75%，在**隔离新模板族 + 更难**上为 37.7% → 印证了「评测必须模板隔离」的必要性，当前模型对难度 3 与部分题型的泛化存在明确缺口；
2. 优先级：① 阶段 3 跑 20k 作对照（预期有限提升，但必须验证）；② 阶段 4 用 GRPO（可验算奖励）直接压制计算错误；③ 数据侧补充难度 3 与模板多样性样本；
3. 评测基建已就绪：`validate.py`（断点续跑 + 分题型统计）、`rescore.py`（等价重评分）、`diag_eval.py`（难度诊断）。

产物：`qlora_math/outputs/validate/5k_on_eval300.{samples,rescored}.jsonl`、`.summary.json`

## 📊 阶段 3：20k 训练与对比结果（2026-10-16）

### 训练指标（20k 全量 × 2 epochs）

| 指标 | 5k | 20k |
|------|-----|-----|
| 步数 / 耗时 | 626 步 / 20.5 min | 2500 步 / 81.8 min |
| Loss | 1.246 → 0.0057 | 1.391 → 0.0051 |
| 峰值显存 | 8.50 GiB | 8.50 GiB |
| 可训练参数 | 43.3 M（0.78%） | 43.3 M（0.78%） |

### 隔离集（300 题）对比：数据 4 倍 ≠ 泛化提升 ⚠️

| 指标 | 5k-LoRA | 20k-LoRA | 结论 |
|------|---------|----------|------|
| 格式合规率 | 100% | 100% | 协议学习稳定迁移 |
| **等价正确率** | **37.7%（113/300）** | **37.7%（113/300）** | **完全一致** |
| 难度 2 | 48.9% | 50.0% | ±1.1pp（噪声内） |
| 难度 3 | 18.8% | 17.0% | ±1.8pp（噪声内） |
| 逐题翻转 | — | 18 条改善 / 18 条变差 | **净变化 0** |

分题型变化（每类 n=10–11，±2 属二项噪声）：`integral_by_parts` −4、`improper_integral` −2、`ode_exponential_growth` +2、`ode_first_order` +2、`tangent_line` +2 等——**正负相抵，无显著差异**。

### 结论（重要负结果）

1. **同分布数据量提升（5k→20k）对模板隔离集泛化零增益**。瓶颈不在数据量，而在：
   - ① 模板族多样性：5k/20k 同为 29 类模式，20k 只是同模板的更多参数组合；
   - ② 难度覆盖：训练数据仅难度 1–2，评测含难度 3；
   - ③ 推理时的计算可靠性（退化解法之外的算术滑移）。
2. 协议能力（100% 合规）与分布内解题模式全部稳定，证实微调本身的"格式对齐"目标达成；
3. **阶段 4 方向已明确**：
   - 数据侧：新增模板族、难度 3 样本、含显式验算步骤的样本（重点针对 9 类 0% 题型）；
   - 算法侧：GRPO（`canonical_answer` + math_verify 作可验证奖励）直接优化正确率；
   - 评测侧：固定隔离集，逐版本对比（5k / 20k / 后续版本）。

产物：`qlora_math/outputs/20k/adapter`（与 5k 版在隔离集上等价，可作为阶段 4 的起点）；对照汇总：`outputs/validate/20k_on_eval300.summary.json`

## 📊 v3 实验结果（sft_v3 训练 + 双基准评测，2026-10-16）

### 数据与训练
- **数据**：`data/sft_v3/`（20,000 = A 复用 6k + B 新结构 9.5k + C 难度 3 共 4.5k；全部含分步推导 + 显式验算段；独立质检全过 ✅ 文件哈希/格式/隔离与声明完全一致）；
- **训练**：2500 步（2 epochs）/ 93 分钟 / loss 1.755 → 0.011 / 峰值显存 8.74 GiB；
- **适配器**：`qlora_math/outputs/v3/adapter`。

### 主指标：eval_v3 —— 🎉 100%

| 指标 | 结果 |
|------|------|
| 格式合规率 | 100% |
| **等价正确率** | **100.0%（300/300）**（严格 99.3%，math_verify 补 2 条等价写法） |
| 分题型 | **20 个重点题型全部 15/15**（此前 0% 的 9 类全部满分） |

### 历史基准：iso300 三模型对比

| 指标 | 5k | 20k | v3 |
|------|-----|-----|-----|
| 总等价正确率 | 37.7% | 37.7% | 34.3% |
| 难度 2 | 48.9% | 50.0% | 47.3% |
| 难度 3 | 18.8% | 17.0% | 12.5% |

**类别间大幅再分配**（v3 vs 20k，仅列 |Δ|≥2）：
- ⬆️ 提升：`integral_by_parts` 0→8、`improper_integral` 5→10、`limit_factor` 0→2；
- ⬇️ 回退：`integral_substitution` 10→1（−9）、`quadratic_multivariable_minimum` 8→3（−5）、`ode_second_order` 7→2（−5）。

### 解读（重要）

1. **v3 的设计目标完全达成**：新结构数据把 9 类零分题型全部教通；eval_v3 已做结构签名级隔离（独立复核 0 重合）→ 属于**真实的族内泛化**；
2. **100% 的边界**：eval_v3 是生成器的"结构兄弟"（签名不同、风格同源）且全为难度 2——证明"掌握 v3 生成器的结构空间"，**不等于开放世界泛化**；后续评测应加入难度 3 与跨族题目；
3. **旧强项出现明显遗忘**：A 组回放仅 6k 且偏向弱项，`substitution / ode2 / quad_min` 等旧强项在新数据挤压下退化——续训分布偏移的经典代价；
4. **净效应取决于部署目标**：面向 v3 同族题目 → 巨幅提升（约 35% → 100%）；面向任意旧族题目 → 约持平略降。

### 下一步：v3.1（均衡回放修复遗忘）

- **方案**：v3.1 = v3 的 B/C 新结构（14k，成绩来源） + **全量旧 20k 均衡回放**（29 类各约 690 条）→ 40k 数据，1–2 epochs（约 90–170 分钟）；
- **验收**：eval_v3 ≥ 95%（不丢新能力）且 iso300 ≥ 40%（召回旧能力），难度 3 ≥ 25%；
- 之后接 **GRPO**（`canonical_answer` + math_verify 可验证奖励）做最后一轮压榨。

产物：`outputs/validate/v3_on_eval_v3_300.*`、`v3_on_eval300.*`；对比脚本 `compare_v3.py`